In [3]:
# Statistical testing helps us verify whether observed
# patterns are real or simply due to random variation.

import pandas as pd
import numpy as np

from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
from scipy.stats import f_oneway

from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", None)

# Load the final feature-engineered dataset
df = pd.read_parquet("../data/processed/featured_data.parquet")

print("Rows:", len(df))
print("Users:", df["user_id"].nunique())

Rows: 4937815
Users: 376483


# Statistical Hypothesis Testing

Exploratory analysis revealed several behavioral patterns in the dataset.
This notebook validates those findings using statistical hypothesis testing.

Goals:

- Verify whether weekend behavior differs from weekdays
- Determine whether buyers behave differently from non-buyers
- Test whether conversion changes across time periods
- Measure the practical importance of observed differences
- Support business recommendations with statistical evidence

## Test 1: Weekend vs Weekday Behavior

### Business Question

Do users behave differently on weekends compared to weekdays?

### Hypotheses

H₀: User behavior is independent of weekend status.

H₁: User behavior changes between weekdays and weekends.

### Why This Matters

If user behavior differs substantially on weekends,
marketing campaigns and promotions may need different strategies.

In [4]:
# Compare action distributions between weekdays and weekends

contingency = pd.crosstab(
    df["is_weekend"],
    df["behavior_type"]
)

contingency

behavior_type,buy,cart,fav,pv
is_weekend,,,,
0,54563,142654,74979,2321073
1,45490,131589,67615,2099852


In [5]:
# Chi-Square tests whether two categorical variables
# are associated with each other.

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-value: {p_value:.6f}")

Chi-Square Statistic: 191.34
P-value: 0.000000


In [6]:
# Statistical significance alone is not enough.
# Cramer's V measures the strength of the relationship.

n = contingency.values.sum()

cramers_v = np.sqrt(
    chi2 /
    (
        n *
        (min(contingency.shape) - 1)
    )
)

print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.0062


### Interpretation

The Chi-Square test produced a statistically significant result
(χ² = 191.34, p < 0.001), indicating that user behavior differs
between weekdays and weekends.

However, the observed effect size is extremely small
(Cramer's V = 0.0062). While the difference is statistically
detectable due to the very large dataset size, the practical
impact on business decision-making is minimal.

### Business Takeaway

Although weekday and weekend users exhibit slightly different
behavior patterns, the difference is too small to justify
separate platform strategies solely based on weekend status.

Other factors such as user segment, product category, and
time of day are likely to have a much stronger influence
on conversion behavior.

In [7]:
# Final conclusion for Test 1

if p_value < 0.05:
    print("Reject H0")
    print("User behavior differs between weekdays and weekends.")
else:
    print("Fail to Reject H0")
    print("No evidence of behavioral differences.")

Reject H0
User behavior differs between weekdays and weekends.


| Metric                   |                       Result |
| ------------------------ | ---------------------------: |
| Test                     | Chi-Square Independence Test |
| χ² Statistic             |                       191.34 |
| P-value                  |                      < 0.001 |
| Cramer's V               |                       0.0062 |
| Statistical Significance |                          Yes |
| Practical Significance   |                   Negligible |
| Decision                 |                    Reject H₀ |


# Test 2: Hour Segment vs Conversion

### Business Question

Does purchase behavior change across different periods of the day?

### Why This Matters

Understanding when users are most likely to purchase helps
businesses schedule promotions, advertisements, push notifications,
and remarketing campaigns more effectively.

### Hypotheses

H₀: Conversion is independent of hour segment.

H₁: Conversion differs across hour segments.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramer's V

In [8]:
# Create a binary conversion indicator.
# Purchase events are treated as successful conversions.

df["converted"] = (
    df["behavior_type"] == "buy"
).astype(int)

print(df["converted"].value_counts())

converted
0    4837762
1     100053
Name: count, dtype: int64


In [9]:
# Compare conversion outcomes across different
# periods of the day.

contingency = pd.crosstab(
    df["hour_segment"],
    df["converted"]
)

contingency

converted,0,1
hour_segment,,
Afternoon,936841,15326
Evening,115173,1571
Late Night,1324342,32605
Lunch,767509,13855
Morning,1525173,34154
Night,168724,2542


In [10]:
# Run Chi-Square Test of Independence

from scipy.stats import chi2_contingency
import numpy as np

chi2, p_value, dof, expected = chi2_contingency(
    contingency
)

print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of Freedom: {dof}")

Chi-Square Statistic: 2791.24
P-value: 0.000000
Degrees of Freedom: 5


In [11]:
# Measure the practical strength of the relationship.
# Large datasets can produce significant p-values even
# when the actual effect is very small.

n = contingency.values.sum()

cramers_v = np.sqrt(
    chi2 /
    (
        n *
        (min(contingency.shape) - 1)
    )
)

print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.0238


In [12]:
# Summarize the statistical decision.

if p_value < 0.05:
    print("Reject H0")
    print("Conversion behaviour differs across hour segments.")
else:
    print("Fail to Reject H0")
    print("No significant difference detected.")

Reject H0
Conversion behaviour differs across hour segments.


### Interpretation

The Chi-Square test was conducted to determine whether conversion
behaviour varies across different periods of the day.

The test produced a statistically significant result
(χ² = 2791.24, p < 0.001), indicating that conversion behaviour
is not evenly distributed across hour segments.

However, the observed effect size is relatively small
(Cramer's V = 0.0238). This suggests that while time of day
does influence conversion behaviour, it is not the sole driver
of purchasing decisions.

The large sample size allows even subtle behavioural differences
to be detected statistically.

### Business Takeaway

Time of day has a measurable impact on conversion behaviour.

This finding supports the earlier exploratory analysis, which
showed that late-night and early-morning users exhibited higher
conversion rates than users active during evening hours.

Businesses can leverage these insights by scheduling promotions,
retargeting campaigns, and customer engagement activities during
high-intent periods to improve marketing effectiveness.

| Metric | Result |
|----------|----------:|
| Test | Chi-Square Independence Test |
| Variable 1 | Hour Segment |
| Variable 2 | Conversion |
| Chi-Square Statistic | 2791.24 |
| Degrees of Freedom | 5 |
| P-value | < 0.001 |
| Cramer's V | 0.0238 |
| Statistical Significance | Yes |
| Practical Significance | Small |
| Decision | Reject H₀ |

# Test 3: Buyer vs Non-Buyer Session Duration

### Business Question

Do users who eventually make a purchase spend more time
engaging with the platform than users who never purchase?

### Why This Matters

Session duration is a strong indicator of user engagement.

If buyers consistently spend more time on the platform,
businesses can focus on increasing engagement through
better recommendations, product discovery, and user experience.

### Hypotheses

H₀: Buyer and non-buyer sessions have the same average duration.

H₁: Buyer and non-buyer sessions have different average durations.

### Statistical Test

Welch's T-Test

### Effect Size

Cohen's d

In [2]:
import pandas as pd
import numpy as np

from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
from scipy.stats import f_oneway

from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet(
    "../data/processed/featured_data.parquet"
)

print(df.shape)

(4937815, 23)


In [13]:
# Build a session-level dataset for analysis

session_df = (
    df.groupby("session_id")
      .agg(
          user_id=("user_id", "first"),
          session_duration=("time_diff", "sum")
      )
      .reset_index()
)

print(session_df.shape)

session_df.head()

(2753235, 3)


,session_id,user_id,session_duration
0,1000001_S1,1000001,87.0
1,1000001_S2,1000001,42465.0
2,1000001_S3,1000001,337111.0
3,1000001_S4,1000001,2694.0
4,1000004_S1,1000004,0.0


In [14]:
# Identify users who have made at least one purchase

buyers = set(
    df[
        df["behavior_type"] == "buy"
    ]["user_id"]
)

# Mark each session as belonging to a buyer or non-buyer

session_df["converted"] = (
    session_df["user_id"]
      .isin(buyers)
      .astype(int)
)

session_df["converted"].value_counts()

converted
0    2031352
1     721883
Name: count, dtype: int64

In [15]:
# Separate session durations into buyer and non-buyer groups

buyer_sessions = session_df[
    session_df["converted"] == 1
]["session_duration"]

nonbuyer_sessions = session_df[
    session_df["converted"] == 0
]["session_duration"]

print("Buyer Sessions:", len(buyer_sessions))
print("Non-Buyer Sessions:", len(nonbuyer_sessions))

Buyer Sessions: 721883
Non-Buyer Sessions: 2031352


In [16]:
# Compare average session duration between the two groups

buyer_sessions.describe()

count    721883.000000
mean      58882.599801
std       80725.833123
min           0.000000
25%        6085.000000
50%       30118.000000
75%       80253.500000
max      738059.000000
Name: session_duration, dtype: float64

In [17]:
nonbuyer_sessions.describe()

count    2.031352e+06
mean     6.959024e+04
std      9.691705e+04
min      0.000000e+00
25%      5.400000e+03
50%      3.447700e+04
75%      8.847500e+04
max      7.443210e+05
Name: session_duration, dtype: float64

In [18]:
# Welch's T-Test is used because the two groups
# may have different variances and sample sizes.

from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    buyer_sessions,
    nonbuyer_sessions,
    equal_var=False
)

print(f"T-Statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.6f}")

T-Statistic: -91.64
P-value: 0.000000


In [19]:
# Cohen's d measures practical significance.

import numpy as np

pooled_std = np.sqrt(
    (
        buyer_sessions.var()
        +
        nonbuyer_sessions.var()
    ) / 2
)

cohens_d = (
    buyer_sessions.mean()
    -
    nonbuyer_sessions.mean()
) / pooled_std

print(f"Cohen's d: {cohens_d:.3f}")

Cohen's d: -0.120


In [20]:
# Summarize the statistical decision

if p_value < 0.05:
    print("Reject H0")
    print("Session duration differs significantly between buyers and non-buyers.")
else:
    print("Fail to Reject H0")
    print("No significant difference detected.")

Reject H0
Session duration differs significantly between buyers and non-buyers.


In [21]:
# Compare average session duration

comparison = pd.DataFrame({
    "Group": ["Buyer", "Non-Buyer"],
    "Average Session Duration (sec)": [
        buyer_sessions.mean(),
        nonbuyer_sessions.mean()
    ]
})

comparison

,Group,Average Session Duration (sec)
0,Buyer,58882.599801
1,Non-Buyer,69590.241826


### Interpretation

A Welch T-Test was conducted to compare session duration
between buyers and non-buyers.

The test produced a statistically significant result
(t = -91.64, p < 0.001), indicating that the average session
duration differs between the two groups.

However, the effect size is small
(Cohen's d = -0.120), suggesting that the practical difference
between the groups is limited.

Interestingly, non-buyers spend more time on the platform
(mean = 69,590 seconds) than buyers
(mean = 58,883 seconds).

This finding suggests that longer browsing sessions do not
necessarily lead to purchases. Buyers appear to make decisions
more efficiently, whereas non-buyers may spend additional time
exploring products without completing a transaction.

### Business Takeaway

Increasing session duration alone is unlikely to improve
conversion rates.

Instead, efforts should focus on reducing purchase friction,
improving product discovery, strengthening recommendations,
and simplifying the checkout process.

The results indicate that decision quality may be more
important than browsing duration in driving purchases.

| Metric | Result |
|----------|----------:|
| Test | Welch T-Test |
| Buyer Mean Session Duration | 58,883 sec |
| Non-Buyer Mean Session Duration | 69,590 sec |
| T-Statistic | -91.64 |
| P-value | < 0.001 |
| Cohen's d | -0.120 |
| Statistical Significance | Yes |
| Practical Significance | Small |
| Decision | Reject H₀ |

# Test 4: Buyer vs Non-Buyer Session Count

### Business Question

Do buyers return to the platform more frequently than non-buyers?

### Why This Matters

Repeated visits often indicate stronger purchase intent.

Understanding whether buyers engage through multiple sessions
can help businesses design retargeting campaigns, loyalty
programs, and customer retention strategies.

### Hypotheses

H₀: Buyers and non-buyers have the same average number of sessions.

H₁: Buyers and non-buyers have different average numbers of sessions.

### Statistical Test

Welch's T-Test

### Effect Size

Cohen's d

In [22]:
# Count the number of unique sessions generated by each user

session_counts = (
    df.groupby("user_id")["session_id"]
      .nunique()
)

session_counts.head()

user_id
1     8
3     7
4    14
5     5
6    11
Name: session_id, dtype: int64

In [23]:
# Identify users who made at least one purchase

buyers = set(
    df[
        df["behavior_type"] == "buy"
    ]["user_id"]
)

# Split users into buyer and non-buyer groups

buyer_session_counts = session_counts[
    session_counts.index.isin(buyers)
]

nonbuyer_session_counts = session_counts[
    ~session_counts.index.isin(buyers)
]

print("Buyer Users:", len(buyer_session_counts))
print("Non-Buyer Users:", len(nonbuyer_session_counts))

Buyer Users: 78609
Non-Buyer Users: 297874


In [24]:
# Examine the distribution of session counts

buyer_session_counts.describe()

count    78609.000000
mean         9.183211
std          5.680547
min          1.000000
25%          5.000000
50%          8.000000
75%         12.000000
max         46.000000
Name: session_id, dtype: float64

In [25]:
nonbuyer_session_counts.describe()

count    297874.000000
mean          6.819501
std           4.948373
min           1.000000
25%           3.000000
50%           6.000000
75%           9.000000
max          58.000000
Name: session_id, dtype: float64

In [26]:
# Compare session counts between buyers and non-buyers

from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    buyer_session_counts,
    nonbuyer_session_counts,
    equal_var=False
)

print(f"T-Statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.6f}")

T-Statistic: 106.49
P-value: 0.000000


In [27]:
# Measure practical significance using Cohen's d

pooled_std = np.sqrt(
    (
        buyer_session_counts.var()
        +
        nonbuyer_session_counts.var()
    ) / 2
)

cohens_d = (
    buyer_session_counts.mean()
    -
    nonbuyer_session_counts.mean()
) / pooled_std

print(f"Cohen's d: {cohens_d:.3f}")

Cohen's d: 0.444


In [28]:
# Compare average session counts

comparison = pd.DataFrame({
    "Group": ["Buyer", "Non-Buyer"],
    "Average Sessions": [
        buyer_session_counts.mean(),
        nonbuyer_session_counts.mean()
    ]
})

comparison

,Group,Average Sessions
0,Buyer,9.183211
1,Non-Buyer,6.819501


### Interpretation

A Welch T-Test was conducted to compare the number of sessions
generated by buyers and non-buyers.

The test produced a statistically significant result
(t = 106.49, p < 0.001), indicating that buyers and non-buyers
exhibit different visitation patterns.

Buyers generated an average of 9.18 sessions, while non-buyers
generated an average of 6.82 sessions.

The observed effect size was moderate
(Cohen's d = 0.444), suggesting that the difference is not only
statistically significant but also meaningful from a business
perspective.

Unlike the session duration analysis, where the effect size was
small, session frequency shows a substantially stronger
relationship with purchasing behavior.

### Business Takeaway

Users who return to the platform more frequently are
significantly more likely to make a purchase.

This suggests that repeat engagement is a stronger predictor
of conversion than simply spending more time during a single
session.

Businesses should focus on strategies that encourage users
to revisit the platform, such as:

- Personalized recommendations
- Remarketing campaigns
- Wishlist reminders
- Push notifications
- Loyalty programs

Increasing return visits may have a direct positive impact
on conversion performance.

| Metric | Result |
|----------|----------:|
| Test | Welch T-Test |
| Buyer Mean Sessions | 9.18 |
| Non-Buyer Mean Sessions | 6.82 |
| T-Statistic | 106.49 |
| P-value | < 0.001 |
| Cohen's d | 0.444 |
| Statistical Significance | Yes |
| Practical Significance | Moderate |
| Decision | Reject H₀ |

# Test 5: Product Category vs Conversion

### Business Question

Does product category influence conversion behaviour?

### Why This Matters

Not all product categories perform equally.

Some categories may naturally attract buyers,
while others generate large amounts of browsing
activity with relatively few purchases.

Understanding category-level differences helps
businesses prioritize inventory, promotions,
recommendation systems, and marketing spend.

### Hypotheses

H₀: Conversion is independent of product category.

H₁: Conversion differs across product categories.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramer's V

In [29]:
# Working with every category would create an
# extremely sparse contingency table.

# Focus on the 20 most active categories.

top_categories = (
    df["category_id"]
      .value_counts()
      .head(20)
      .index
)

top_categories

Index([4756105, 4145813, 2355072, 3607361,  982926, 2520377, 4801426, 1320293,
       2465336, 3002561, 2735466,  149192, 4181361, 2885642, 4217906, 1080785,
        154040, 3738615, 1879194, 2640118],
      dtype='int64', name='category_id')

In [30]:
# Create a filtered dataset containing only
# the top categories.

category_df = df[
    df["category_id"].isin(top_categories)
].copy()

print(category_df.shape)

(1853557, 24)


In [31]:
# Create a conversion indicator

category_df["converted"] = (
    category_df["behavior_type"] == "buy"
).astype(int)

category_df["converted"].value_counts()

converted
0    1836385
1      17172
Name: count, dtype: int64

In [32]:
# Compare conversion outcomes across categories

contingency = pd.crosstab(
    category_df["category_id"],
    category_df["converted"]
)

contingency

converted,0,1
category_id,,
149192,54469,587
154040,46888,115
982926,149817,1221
1080785,50478,189
1320293,96040,849
1879194,40973,682
2355072,164813,661
2465336,80991,693
2520377,107508,511


In [33]:
# Run Chi-Square Test

from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(
    contingency
)

print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of Freedom: {dof}")

Chi-Square Statistic: 7977.68
P-value: 0.000000
Degrees of Freedom: 19


In [34]:
# Measure practical significance

import numpy as np

n = contingency.values.sum()

cramers_v = np.sqrt(
    chi2 /
    (
        n *
        (min(contingency.shape) - 1)
    )
)

print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.0656


In [35]:
# Final statistical decision

if p_value < 0.05:
    print("Reject H0")
    print("Conversion behaviour differs across categories.")
else:
    print("Fail to Reject H0")
    print("No significant category effect detected.")

Reject H0
Conversion behaviour differs across categories.


In [36]:
# Calculate category-level conversion rates

category_summary = (
    category_df.groupby("category_id")
               .agg(
                   views=("behavior_type",
                          lambda x: (x=="pv").sum()),
                   purchases=("behavior_type",
                              lambda x: (x=="buy").sum())
               )
)

category_summary["conversion_rate"] = (
    category_summary["purchases"]
    /
    category_summary["views"]
    * 100
)

category_summary.sort_values(
    "conversion_rate",
    ascending=False
)

,views,purchases,conversion_rate
category_id,,,
2885642,47277,1601,3.386425
2735466,55156,1703,3.087606
2640118,35866,920,2.565103
1879194,37669,682,1.810507
4217906,45420,739,1.627037
4801426,91993,1335,1.451197
3738615,41359,496,1.199255
149192,49389,587,1.188524
3002561,70253,816,1.161516


### Interpretation

A Chi-Square Test of Independence was conducted to determine
whether conversion behaviour varies across product categories.

The test produced a statistically significant result
(χ² = 7977.68, p < 0.001), indicating that conversion behaviour
is not independent of product category.

The effect size measured using Cramer's V was 0.0656,
which is substantially larger than the values observed in the
weekend and hour-segment analyses.

Although the effect is still considered small, it represents
the strongest categorical relationship identified so far.

This suggests that product category plays a more important role
in influencing purchasing behaviour than factors such as
weekday/weekend status or time of day.

### Business Takeaway

Product category has a measurable influence on conversion.

Certain categories consistently attract users who are more likely
to purchase, while others generate browsing activity without
resulting in comparable sales.

Marketing resources, recommendation systems, inventory planning,
and promotional campaigns should prioritize categories with
historically higher conversion rates.

Category-level optimization is likely to generate a greater
business impact than broad scheduling changes based solely on
time or day.

| Metric | Result |
|----------|----------:|
| Test | Chi-Square Independence Test |
| Variable 1 | Category ID |
| Variable 2 | Conversion |
| Chi-Square Statistic | 7977.68 |
| Degrees of Freedom | 19 |
| P-value | < 0.001 |
| Cramer's V | 0.0656 |
| Statistical Significance | Yes |
| Practical Significance | Small to Moderate |
| Decision | Reject H₀ |

# Test 6: Conversion Across Hour Segments

### Business Question

Do different periods of the day generate different conversion rates?

### Why This Matters

User purchase intent often varies throughout the day.

Identifying high-converting time periods can help optimize:

- Promotional campaigns
- Push notifications
- Retargeting efforts
- Advertising schedules

### Hypotheses

H₀: Average conversion rates are equal across all hour segments.

H₁: At least one hour segment has a different conversion rate.

### Statistical Test

One-Way ANOVA

In [37]:
# Create a binary conversion indicator.
# Purchase events are treated as conversions.

df["converted"] = (
    df["behavior_type"] == "buy"
).astype(int)

df["converted"].value_counts()

converted
0    4837762
1     100053
Name: count, dtype: int64

In [38]:
# Examine average conversion rates across
# different periods of the day.

hour_segment_summary = (
    df.groupby("hour_segment")
      ["converted"]
      .agg(
          conversion_rate="mean",
          observations="count"
      )
)

hour_segment_summary["conversion_rate"] *= 100

hour_segment_summary.sort_values(
    "conversion_rate",
    ascending=False
)

,conversion_rate,observations
hour_segment,,
Late Night,2.402820,1356947
Morning,2.190304,1559327
Lunch,1.773181,781364
Afternoon,1.609592,952167
Night,1.484241,171266
Evening,1.345679,116744


In [39]:
# Build groups for ANOVA.
# Each group contains conversion observations
# for a specific hour segment.

groups = [
    df[
        df["hour_segment"] == segment
    ]["converted"]
    for segment in df["hour_segment"].unique()
]

In [40]:
# Run One-Way ANOVA

from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print(f"F-Statistic: {f_stat:.2f}")
print(f"P-value: {p_value:.6f}")

F-Statistic: 558.56
P-value: 0.000000


In [41]:


# Statistical decision

if p_value < 0.05:
    print("Reject H0")
    print("Average conversion differs across hour segments.")
else:
    print("Fail to Reject H0")
    print("No significant difference detected.")

Reject H0
Average conversion differs across hour segments.


In [42]:
hour_segment_summary.sort_values(
    "conversion_rate",
    ascending=False
)

,conversion_rate,observations
hour_segment,,
Late Night,2.402820,1356947
Morning,2.190304,1559327
Lunch,1.773181,781364
Afternoon,1.609592,952167
Night,1.484241,171266
Evening,1.345679,116744


### Interpretation

A One-Way ANOVA was conducted to determine whether average
conversion rates differ across hour segments.

The test produced a statistically significant result
(F = 558.56, p < 0.001), indicating that conversion behaviour
is not uniform throughout the day.

The observed differences are substantial enough that they
cannot be explained by random variation alone.

Among all time periods, Late Night exhibited the highest
conversion rate (2.40%), followed by Morning (2.19%).

Evening recorded the lowest conversion rate (1.35%),
while Night also showed relatively weak performance (1.48%).

These findings are consistent with the earlier funnel analysis,
which showed that late-night users demonstrate stronger purchase
intent than users browsing during evening hours.

### Business Takeaway

Time of day has a significant influence on conversion behaviour.

Users browsing during Late Night and Morning periods are
substantially more likely to purchase than users active
during Evening hours.

Businesses can improve marketing efficiency by scheduling:

- Promotional campaigns
- Retargeting advertisements
- Push notifications
- Personalized offers

during high-conversion periods rather than distributing
marketing efforts uniformly throughout the day.

The results suggest that campaign timing should be treated
as an important optimization variable.

| Metric | Result |
|----------|----------:|
| Test | One-Way ANOVA |
| Variable | Hour Segment |
| F-Statistic | 558.56 |
| P-value | < 0.001 |
| Statistical Significance | Yes |
| Decision | Reject H₀ |
| Highest Conversion Segment | Late Night (2.40%) |
| Lowest Conversion Segment | Evening (1.35%) |

# Test 7: Conversion Across Days of Week

### Business Question

Do conversion rates differ across days of the week?

### Why This Matters

User purchasing behaviour may vary depending on the day.

Understanding these patterns helps businesses optimize:

- Campaign scheduling
- Promotional timing
- Inventory planning
- Advertising spend allocation

### Hypotheses

H₀: Average conversion rates are equal across all days of the week.

H₁: At least one day has a different conversion rate.

### Statistical Test

One-Way ANOVA

In [44]:
# Create a binary conversion indicator.
# Purchase events are treated as successful conversions.

df["converted"] = (
    df["behavior_type"] == "buy"
).astype(int)

df["converted"].value_counts()

converted
0    4837762
1     100053
Name: count, dtype: int64

In [45]:
# Calculate average conversion rates for each day.

day_summary = (
    df.groupby("day")
      ["converted"]
      .agg(
          conversion_rate="mean",
          observations="count"
      )
)

day_summary["conversion_rate"] *= 100

day_summary.sort_values(
    "conversion_rate",
    ascending=False
)

,conversion_rate,observations
day,,
Monday,2.186977,499548
Wednesday,2.184433,514733
Thursday,2.129660,527643
Tuesday,2.123527,493236
Sunday,1.988842,1123367
Friday,1.914142,558109
Saturday,1.895545,1221179


In [46]:
# Build groups for ANOVA.
# Each group contains conversion observations
# for a specific day of the week.

groups = [
    df[
        df["day"] == day
    ]["converted"]
    for day in df["day"].unique()
]

In [47]:
# Run One-Way ANOVA

from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print(f"F-Statistic: {f_stat:.2f}")
print(f"P-value: {p_value:.6f}")

F-Statistic: 55.03
P-value: 0.000000


In [48]:
# Statistical decision

if p_value < 0.05:
    print("Reject H0")
    print("Average conversion differs across days of the week.")
else:
    print("Fail to Reject H0")
    print("No significant difference detected.")

Reject H0
Average conversion differs across days of the week.


In [49]:
# View conversion performance by day

day_summary.sort_values(
    "conversion_rate",
    ascending=False
)

,conversion_rate,observations
day,,
Monday,2.186977,499548
Wednesday,2.184433,514733
Thursday,2.129660,527643
Tuesday,2.123527,493236
Sunday,1.988842,1123367
Friday,1.914142,558109
Saturday,1.895545,1221179


### Interpretation

A One-Way ANOVA was conducted to determine whether average
conversion rates differ across days of the week.

The test produced a statistically significant result
(F = 55.03, p < 0.001), indicating that conversion behaviour
is not identical across all days.

The observed differences are unlikely to be caused by random
variation alone.

Monday recorded the highest conversion rate (2.19%), closely
followed by Wednesday (2.18%).

Saturday exhibited the lowest conversion rate (1.90%), while
Friday also showed relatively weak performance (1.91%).

Although the differences are statistically significant, the
overall range of conversion rates across days remains relatively
small compared to the variation observed across user segments
and session frequency.

These findings suggest that day-of-week effects exist, but they
are not the primary drivers of purchasing behaviour.

### Business Takeaway

Weekday users demonstrate slightly stronger purchase intent
than weekend users.

The highest-performing days occur at the beginning and middle
of the week, while weekends generate larger browsing volumes
but lower conversion rates.

Businesses can leverage this insight by:

- Scheduling conversion-focused campaigns on weekdays
- Using weekends for awareness and engagement campaigns
- Launching promotional offers on Monday through Thursday
- Retargeting weekend browsers during the following week

While day-of-week effects are measurable, they should be used
alongside stronger predictors such as user engagement and
product category performance.

| Metric | Result |
|----------|----------:|
| Test | One-Way ANOVA |
| Variable | Day of Week |
| F-Statistic | 55.03 |
| P-value | < 0.001 |
| Statistical Significance | Yes |
| Decision | Reject H₀ |
| Highest Conversion Day | Monday (2.19%) |
| Lowest Conversion Day | Saturday (1.90%) |

# Test 8: Weekend vs Weekday Cart-to-Purchase Conversion

### Business Question

Are users more likely to complete a purchase after adding
an item to their cart on weekends or weekdays?

### Why This Matters

Cart additions represent strong purchase intent.

If cart-to-purchase conversion differs between weekdays and
weekends, businesses can optimize:

- Retargeting campaigns
- Promotional timing
- Cart reminder notifications
- Checkout incentives

### Hypotheses

H₀: Weekend and weekday cart conversion rates are equal.

H₁: Weekend and weekday cart conversion rates are different.

### Statistical Test

Two-Proportion Z-Test

In [50]:
# Count cart and purchase events separately
# for weekdays and weekends.

weekday_cart = len(
    df[
        (df["behavior_type"] == "cart")
        &
        (df["is_weekend"] == 0)
    ]
)

weekday_buy = len(
    df[
        (df["behavior_type"] == "buy")
        &
        (df["is_weekend"] == 0)
    ]
)

weekend_cart = len(
    df[
        (df["behavior_type"] == "cart")
        &
        (df["is_weekend"] == 1)
    ]
)

weekend_buy = len(
    df[
        (df["behavior_type"] == "buy")
        &
        (df["is_weekend"] == 1)
    ]
)

print("Weekday Carts:", weekday_cart)
print("Weekday Purchases:", weekday_buy)

print()

print("Weekend Carts:", weekend_cart)
print("Weekend Purchases:", weekend_buy)

Weekday Carts: 142654
Weekday Purchases: 54563

Weekend Carts: 131589
Weekend Purchases: 45490


In [51]:
# Calculate cart-to-purchase conversion rates.

weekday_conversion = (
    weekday_buy
    /
    weekday_cart
    * 100
)

weekend_conversion = (
    weekend_buy
    /
    weekend_cart
    * 100
)

print(f"Weekday Cart Conversion: {weekday_conversion:.2f}%")
print(f"Weekend Cart Conversion: {weekend_conversion:.2f}%")

Weekday Cart Conversion: 38.25%
Weekend Cart Conversion: 34.57%


In [52]:
# Compare conversion rates statistically.

from statsmodels.stats.proportion import proportions_ztest

count = [
    weekend_buy,
    weekday_buy
]

nobs = [
    weekend_cart,
    weekday_cart
]

z_stat, p_value = proportions_ztest(
    count,
    nobs
)

print(f"Z-Statistic: {z_stat:.2f}")
print(f"P-value: {p_value:.6f}")

Z-Statistic: -19.99
P-value: 0.000000


In [53]:
# Statistical decision

if p_value < 0.05:
    print("Reject H0")
    print("Weekend and weekday cart conversion rates differ.")
else:
    print("Fail to Reject H0")
    print("No significant difference detected.")

Reject H0
Weekend and weekday cart conversion rates differ.


In [54]:
# Create a simple summary table.

summary = pd.DataFrame({
    "Period": ["Weekday", "Weekend"],
    "Cart Events": [weekday_cart, weekend_cart],
    "Purchase Events": [weekday_buy, weekend_buy],
    "Cart Conversion (%)": [
        weekday_conversion,
        weekend_conversion
    ]
})

summary

,Period,Cart Events,Purchase Events,Cart Conversion (%)
0,Weekday,142654,54563,38.248489
1,Weekend,131589,45490,34.569759


### Interpretation

A Two-Proportion Z-Test was conducted to compare cart-to-purchase
conversion rates between weekdays and weekends.

The test produced a statistically significant result
(Z = -19.99, p < 0.001), indicating that the probability of
converting a cart into a purchase differs between the two periods.

Weekday cart conversion was 38.25%, compared to 34.57% on weekends.

The difference of approximately 3.68 percentage points is unlikely
to be caused by random variation and reflects a meaningful change
in purchasing behavior.

These findings suggest that users who add products to their cart
during weekdays are more likely to complete a purchase than users
who add products during weekends.

### Business Takeaway

Weekday users demonstrate stronger purchase intent after adding
items to their cart.

Weekend users appear more likely to browse, compare products,
or postpone purchase decisions.

Businesses can leverage this insight by:

- Increasing cart reminder frequency during weekends
- Offering weekend checkout incentives
- Retargeting weekend cart abandoners on Monday
- Running conversion-focused campaigns during weekdays

Improving weekend cart conversion presents a clear opportunity
to reduce revenue leakage and increase overall sales.

| Metric | Result |
|----------|----------:|
| Test | Two-Proportion Z-Test |
| Comparison | Weekday vs Weekend Cart Conversion |
| Weekday Conversion | 38.25% |
| Weekend Conversion | 34.57% |
| Difference | 3.68 Percentage Points |
| Z-Statistic | -19.99 |
| P-value | < 0.001 |
| Statistical Significance | Yes |
| Decision | Reject H₀ |

# Statistical Findings Summary

Eight statistical tests were conducted to validate key business
hypotheses identified during exploratory analysis.

| Test | Result | Business Interpretation |
|--------|--------|--------|
| Chi-Square: Weekend vs Weekday Behaviour | Significant, negligible effect | Weekend behavior differs slightly but has limited practical impact |
| Chi-Square: Hour Segment vs Conversion | Significant | Time of day influences conversion behavior |
| Welch T-Test: Session Duration | Significant, small effect | Longer sessions do not necessarily lead to purchases |
| Welch T-Test: Session Count | Significant, moderate effect | Repeat visits strongly increase purchase likelihood |
| Chi-Square: Category vs Conversion | Significant | Product category influences conversion outcomes |
| ANOVA: Conversion Across Hour Segments | Significant | Late-night users exhibit the highest conversion rates |
| ANOVA: Conversion Across Days | Significant | Weekday conversion slightly exceeds weekend conversion |
| Proportion Z-Test: Cart Conversion | Significant | Weekday carts convert better than weekend carts |

## Key Business Insights

### 1. Repeat Engagement Drives Conversion

Session count emerged as the strongest behavioral predictor
of purchasing activity. Buyers returned significantly more
often than non-buyers.

### 2. Product Category Matters

Category-level differences had a stronger impact on conversion
than broad temporal factors such as weekdays and weekends.

### 3. Timing Influences Purchase Intent

Late-night and morning users consistently demonstrated higher
conversion rates than evening users.

### 4. Weekend Traffic Converts Less Efficiently

Although weekends generate substantial browsing activity,
weekday users exhibit stronger purchase intent and higher
cart-to-purchase conversion rates.

### 5. Statistical Significance Does Not Always Mean Business Significance

Several tests produced highly significant p-values due to the
large dataset size, but effect-size analysis revealed that some
relationships were practically weak. Both statistical and
practical significance were considered when drawing conclusions.

In [55]:
# Final dataset for SQL and Power BI

df.to_csv(
    "../data/processed/final_ecommerce_data.csv",
    index=False
)

print("Export complete")

Export complete
